# ComfyUI + LTX 2.3 GGUF on Kaggle
**Model:** Unsloth LTX-2.3-GGUF (Q3_K_M)
**Workflows:** T2V, I2V, V2V, ICLoRA, Motion Control
---


In [ ]:
MODEL_VARIANT = "Q3_K_M"
DOWNLOAD_MODELS = True
DOWNLOAD_WORKFLOWS = True
INCLUDE_AUDIO = True
INCLUDE_ICLORA = True
LOW_VRAM_MODE = True
TUNNEL_MODE = "pinggy"
print(f"Config: {MODEL_VARIANT}")

In [ ]:
import os, sys, subprocess, shutil, time
from pathlib import Path
WORKING = Path("/kaggle/working")
COMFY_DIR = WORKING / "ComfyUI"
VENV_DIR = WORKING / "venv"
start = time.time()
print("Installing env...\n")
# FIX: use virtualenv for Python 3.12
if not VENV_DIR.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"], check=True)
    subprocess.run(["virtualenv", str(VENV_DIR), "-p", "/usr/bin/python3.10"], check=True)
python_bin = str(VENV_DIR / "bin" / "python3")
pip = [python_bin, "-m", "pip", "install", "-q"]
subprocess.run([*pip, "torch","torchvision","torchaudio", "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
print("PyTorch OK")
subprocess.run([*pip, "requests","einops","diffusers","accelerate","av","opencv-python"], check=True)
print("Deps OK")
if not COMFY_DIR.exists():
    subprocess.run(["git","clone","--branch","ComfyUI_ltx_2_3_compliant_19_03_2026","https://github.com/Isi-dev/ComfyUI",str(COMFY_DIR)], check=True)
    subprocess.run([*pip, "-r", str(COMFY_DIR / "requirements.txt")], check=True)
os.chdir(str(COMFY_DIR/"custom_nodes"))
for n,u in {"ComfyUI-Manager":"https://github.com/Comfy-Org/ComfyUI-Manager","ComfyUI-GGUF":"https://github.com/city96/ComfyUI-GGUF","ComfyUI-LTXVideo":"https://github.com/Lightricks/ComfyUI-LTXVideo","ComfyUI_KJNodes":"https://github.com/kijai/ComfyUI-KJNodes","rgthree-comfy":"https://github.com/rgthree/rgthree-comfy.git","ComfyUI-VideoHelperSuite":"https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite"}.items():
    p = COMFY_DIR / "custom_nodes" / n
    if not p.exists():
        subprocess.run(["git","clone",u,str(p)],capture_output=True,check=False)
        req = p / "requirements.txt"
        if req.exists(): subprocess.run([*pip,"-r",str(req)],capture_output=True,check=False)
for sd in ["unet","vae","text_encoders","loras"]:
    (COMFY_DIR/"models"/sd).mkdir(parents=True,exist_ok=True)
print(f"Env done ({time.time()-start:.0f}s)")

In [ ]:
import requests, shutil, time
from pathlib import Path
HF = "https://huggingface.co"
MDIR = COMFY_DIR / "models"
MODEL_URLS = {"Q2_K": f"{HF}/unsloth/LTX-2.3-GGUF/resolve/main/ltx-2.3-22b-dev-Q2_K.gguf","Q3_K_M": f"{HF}/unsloth/LTX-2.3-GGUF/resolve/main/ltx-2.3-22b-dev-Q3_K_M.gguf","Q4_K_M": f"{HF}/unsloth/LTX-2.3-GGUF/resolve/main/ltx-2.3-22b-dev-Q4_K_M.gguf"}
def dl(url, dest, label=""):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 1_000_000:
        print(f"  OK {label}"); return
    dest.parent.mkdir(parents=True, exist_ok=True)
    import urllib.request
    print(f"  DL {label}...")
    urllib.request.urlretrieve(url, dest)
    print(f"  OK {dest.stat().st_size/1e9:.1f} GB")
if DOWNLOAD_MODELS:
    t0 = time.time()
    dl(MODEL_URLS[MODEL_VARIANT], MDIR/"unet"/f"ltx-2.3-22b-dev-{MODEL_VARIANT}.gguf", f"Model {MODEL_VARIANT}")
    dl(f"{HF}/unsloth/LTX-2.3-GGUF/resolve/main/vae/ltx-2.3-22b-dev_video_vae.safetensors", MDIR/"vae"/"ltx-2.3-22b-dev_video_vae.safetensors", "VAE")
    dl(f"{HF}/unsloth/LTX-2.3-GGUF/resolve/main/text_encoders/ltx-2.3-22b-dev_embeddings_connectors.safetensors", MDIR/"text_encoders"/"ltx-2.3-22b-dev_embeddings_connectors.safetensors", "Embeddings")
    dl(f"{HF}/unsloth/gemma-3-12b-it-qat-GGUF/resolve/main/gemma-3-12b-it-qat-UD-Q4_K_XL.gguf", MDIR/"text_encoders"/"gemma-3-12b-it-qat-UD-Q4_K_XL.gguf", "Gemma 3 12B")
    dl(f"{HF}/unsloth/gemma-3-12b-it-qat-GGUF/resolve/main/mmproj-BF16.gguf", MDIR/"text_encoders"/"mmproj-BF16.gguf", "mmproj")
    dl(f"{HF}/Lightricks/LTX-2.3/resolve/main/ltx-2.3-22b-distilled-lora-384.safetensors", MDIR/"loras"/"ltx-2.3-22b-distilled-lora-384.safetensors", "LoRA")
    if INCLUDE_ICLORA:
        dl(f"{HF}/Lightricks/LTX-2.3/resolve/main/ltx-2.3-22b-ic-lora-union-control-ref0.5.safetensors", MDIR/"loras"/"ltx-2.3-22b-ic-lora-union-control-ref0.5.safetensors", "IC-LoRA")
    dl(f"{HF}/Lightricks/LTX-2.3/resolve/main/ltx-2.3-spatial-upscaler-x2-1.0.safetensors", MDIR/"latent_upscale_models"/"ltx-2.3-spatial-upscaler-x2-1.0.safetensors", "Upscaler")
    gb = sum(f.stat().st_size for f in MDIR.rglob("*") if f.is_file())/1e9
    print(f"Total: {gb:.1f} GB ({time.time()-t0:.0f}s)")

In [ ]:
import urllib.request, time
from pathlib import Path
wf_dir = Path("/kaggle/working/workflows")
wf_dir.mkdir(exist_ok=True)
WFS = {"LTX-2.3_T2V_I2V_Single_Stage_Distilled_Full.json":"https://raw.githubusercontent.com/Lightricks/ComfyUI-LTXVideo/master/example_workflows/2.3/LTX-2.3_T2V_I2V_Single_Stage_Distilled_Full.json","LTX-2.3_T2V_I2V_Two_Stage_Distilled.json":"https://raw.githubusercontent.com/Lightricks/ComfyUI-LTXVideo/master/example_workflows/2.3/LTX-2.3_T2V_I2V_Two_Stage_Distilled.json","LTX-2.3_ICLoRA_HDR_Distilled.json":"https://raw.githubusercontent.com/Lightricks/ComfyUI-LTXVideo/master/example_workflows/2.3/LTX-2.3_ICLoRA_HDR_Distilled.json","LTX-2.3_ICLoRA_Ingredients_Single_Stage_Distilled.json":"https://raw.githubusercontent.com/Lightricks/ComfyUI-LTXVideo/master/example_workflows/2.3/LTX-2.3_ICLoRA_Ingredients_Single_Stage_Distilled.json","LTX-2.3_ICLoRA_Union_Control_Distilled.json":"https://raw.githubusercontent.com/Lightricks/ComfyUI-LTXVideo/master/example_workflows/2.3/LTX-2.3_ICLoRA_Union_Control_Distilled.json","LTX-2.3_ICLoRA_Motion_Track_Distilled.json":"https://raw.githubusercontent.com/Lightricks/ComfyUI-LTXVideo/master/example_workflows/2.3/LTX-2.3_ICLoRA_Motion_Track_Distilled.json"}
if DOWNLOAD_WORKFLOWS:
    t0 = time.time()
    for n,u in WFS.items():
        d = wf_dir / n
        if d.exists(): continue
        try:
            urllib.request.urlretrieve(u, d)
            print(f"  {n}")
        except Exception as e:
            print(f"  FAIL: {e}")
    print(f"Done ({time.time()-t0:.0f}s)")

In [ ]:
import urllib.request, subprocess, sys, os, time
from pathlib import Path
WORKING = Path("/kaggle/working")
VENV_DIR = WORKING / "venv"
COMFY_DIR = WORKING / "ComfyUI"
python_bin = str(VENV_DIR / "bin" / "python3")
print("GPU:")
subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,index","--format=csv,noheader"])
args = [python_bin, str(COMFY_DIR/"main.py"),"--listen","127.0.0.1","--port","8188"]
if LOW_VRAM_MODE: args.append("--highvram")
subprocess.Popen(args, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("ComfyUI started")
for i in range(30):
    time.sleep(2)
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system/stats", timeout=3)
        print(f"ComfyUI ready! ({i*2}s)"); break
    except: pass
# Pinggy tunnel
comfy_cmd = ' '.join(args)
import subprocess as sp
!wget -q https://raw.githubusercontent.com/wandaweb/jupyter-webui-tunneling/main/pinggy.py -O /kaggle/working/pinggy.py
import IPython; IPython.get_ipython().run_line_magic("cd", "/kaggle/working")
!python /kaggle/working/pinggy.py --command='{comfy_cmd}' --port=8188